In [1]:
# Cell 0 — Path setup
import sys, os
from pathlib import Path
REPO_ROOT = Path.cwd()
if REPO_ROOT.name in ("notebooks", "current notebooks") or not (REPO_ROOT / "scripts").is_dir():
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT = {REPO_ROOT}")

REPO_ROOT = /Users/JanRovirosaIlla/DeepMarkovResearch


In [2]:
# Cell 1 — Imports
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from sklearn.feature_selection import mutual_info_classif

from scripts.config import make_config
from scripts.data import load_master_dataset, preprocess_features, build_all_splits
from scripts.bins import compute_X_t, build_all_configs
from scripts.models import StateConditionedNet
from scripts.train import (
    train_one_run, build_loaders, MasterDataset,
    cache_model, load_cached_model, is_cached,
)
from scripts.eval import evaluate_model, evaluate_baselines
from scripts.plotting import save_fig

print("Imports OK")

Imports OK


In [3]:
# Cell 2 — Experiment configuration
cfg = make_config()   # hyperparameters only (lr, wd, hidden_dims, etc.)

# Fixed experiment config
H        = 10
N_BINS   = 55
N_FEAT   = 30          # top-MI features
SEEDS    = [0, 1, 2, 3, 4]
FA_SEED  = 0           # single seed for feature ablation
FA_SIZES = [15, 30, 50, 194]

# Fixed output dirs (not date-stamped)
RESULTS_DIR = REPO_ROOT / "results" / "optimal_setup"
CACHE_DIR   = RESULTS_DIR / "cache"
FIG_DIR     = RESULTS_DIR / "figures"
for d in [RESULTS_DIR, CACHE_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")
print(f"RESULTS_DIR = {RESULTS_DIR}")

DEVICE = cpu
RESULTS_DIR = /Users/JanRovirosaIlla/DeepMarkovResearch/results/optimal_setup


---
## Section 1 — Data Loading

Load dataset, normalise features on train split, build horizon-10 state/label arrays.
Use `c["idx_train"]` / `c["idx_val"]` / `c["idx_test"]` from the config object to ensure
indices are aligned to the horizon-10 label array `c["y_all"]`.

In [4]:
# Section 1 — Data Loading
prices, F_raw, feature_cols = load_master_dataset("dataset")
splits   = build_all_splits(prices, [H])

# Fit normaliser on splits idx_train (pre-alignment)
F_normed = preprocess_features(F_raw, splits[H]["idx_train"])

train_end = splits[H]["idx_train"][-1] + 1
X_t_all, N_XT, edges_xt = compute_X_t(prices, cfg.n_xt_target, train_end)

configs = build_all_configs(
    prices, F_normed, X_t_all,
    horizons=[H], n_bins_list=[N_BINS],
    N_XT=N_XT, edges_xt=edges_xt, splits=splits,
    sigma_anchor=cfg.sigma_anchor,
)
c = configs[(H, N_BINS)]
N_actual = c["N_actual"]

# CRITICAL: use horizon-aligned indices from c
idx_train = c["idx_train"]
idx_val   = c["idx_val"]
idx_test  = c["idx_test"]

print(f"Splits: train={len(idx_train)}, val={len(idx_val)}, test={len(idx_test)}")
print(f"N_actual={N_actual}, N_XT={N_XT}")
print(f"y_all shape: {c['y_all'].shape},  X_t_all shape: {X_t_all.shape}")
print(f"sigma={c['sigma']:.4f}")

Splits: train=1650, val=354, test=354
N_actual=55, N_XT=55
y_all shape: (2358,),  X_t_all shape: (2368,)
sigma=1.0000


---
## Section 2 — Feature Selection via Mutual Information

Rank all 194 features by MI with the horizon-10 label **on training data only**.
Select top-30 to reduce dimensionality and potential overfitting.

In [5]:
# Section 2 — Feature Selection
X_train_mi = F_normed[idx_train]       # (n_train, 194) — aligned train only
y_train_mi = c["y_all"][idx_train]     # (n_train,) aligned binned labels

print(f"Computing MI on {X_train_mi.shape[0]} train samples × {X_train_mi.shape[1]} features...")
mi_scores = mutual_info_classif(X_train_mi, y_train_mi, random_state=0, n_neighbors=5)
ranking   = np.argsort(mi_scores)[::-1]   # descending MI
sel_idx   = ranking[:N_FEAT]              # top-30 indices
F_sub     = F_normed[:, sel_idx]          # (T_all, 30) — used for all splits

# Save
np.save(RESULTS_DIR / "selected_feature_indices.npy", sel_idx)

print(f"\nTop-{N_FEAT} feature indices: {sel_idx.tolist()}")
print(f"Top-{N_FEAT} MI scores: {mi_scores[sel_idx].round(4).tolist()}")
print(f"F_sub shape: {F_sub.shape}")

Computing MI on 1650 train samples × 194 features...



Top-30 feature indices: [94, 84, 23, 123, 177, 54, 147, 145, 143, 16, 184, 185, 95, 179, 97, 188, 31, 52, 116, 47, 30, 39, 131, 101, 176, 96, 191, 60, 104, 76]
Top-30 MI scores: [0.1733, 0.1699, 0.1687, 0.1674, 0.1666, 0.1657, 0.162, 0.1614, 0.16, 0.1585, 0.158, 0.1569, 0.1559, 0.1557, 0.1555, 0.1552, 0.1551, 0.1545, 0.1536, 0.1527, 0.1522, 0.1521, 0.1521, 0.1519, 0.1519, 0.1517, 0.151, 0.15, 0.1498, 0.1494]
F_sub shape: (2369, 30)


---
## Section 3 — Multi-Seed Training (Stability Check)

Train `StateConditionedNet` with N_FEAT=30 features across 5 seeds.
Record train/val/test NLL per seed. Models are cached for reproducibility.

In [6]:
# Section 3 — Multi-Seed Training
uniform_nll = float(np.log(N_actual))
print(f"Uniform NLL (sanity) = {uniform_nll:.4f}  (expect ~{np.log(55):.2f} for N=55)\n")

seed_rows = []
for seed in SEEDS:
    # Full reproducibility
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    cache_path = CACHE_DIR / f"seed{seed}_h{H}_N{N_BINS}_nfeat{N_FEAT}.pt"
    m = StateConditionedNet(
        n_feat=N_FEAT, n_xt_states=N_XT, n_output=N_actual,
        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout,
    )
    tr_loader, val_loader, te_loader = build_loaders(
        c, F_sub, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    if is_cached(cache_path):
        print(f"  seed={seed}: loading from cache...")
        load_cached_model(m, cache_path)
    else:
        print(f"  seed={seed}: training...")
        best_state, _ = train_one_run(
            m, tr_loader, val_loader, N_actual, c["sigma"], DEVICE,
            lr=cfg.lr, weight_decay=cfg.weight_decay,
            max_epochs=cfg.max_epochs, patience=cfg.patience,
            grad_clip=cfg.grad_clip,
        )
        cache_model(best_state, cache_path)

    m.to(DEVICE)
    m.eval()

    # Evaluate on all three splits (train eval uses aligned idx_train)
    tr_eval = DataLoader(
        MasterDataset(F_sub, X_t_all, c["y_all"], idx_train),
        batch_size=cfg.batch_eval, shuffle=False,
    )
    res_tr  = evaluate_model(m, tr_eval,    N_actual, DEVICE)
    res_val = evaluate_model(m, val_loader, N_actual, DEVICE)
    res_te  = evaluate_model(m, te_loader,  N_actual, DEVICE)

    row = {
        "seed":      seed,
        "train_nll": -res_tr["mean_ll"],
        "val_nll":   -res_val["mean_ll"],
        "test_nll":  -res_te["mean_ll"],
    }
    seed_rows.append(row)
    print(f"  seed={seed}: train={row['train_nll']:.4f} "
          f"val={row['val_nll']:.4f} test={row['test_nll']:.4f}")

df_seeds = pd.DataFrame(seed_rows)
df_seeds.to_csv(RESULTS_DIR / "seed_stability.csv", index=False)
print("\n", df_seeds.to_string(index=False))
print(f"\nmean_test_nll = {df_seeds['test_nll'].mean():.4f}")
print(f"std_test_nll  = {df_seeds['test_nll'].std():.4f}")

Uniform NLL (sanity) = 4.0073  (expect ~4.01 for N=55)

  seed=0: training...


  seed=0: train=3.8835 val=3.9942 test=4.0060
  seed=1: training...


  seed=1: train=3.9042 val=3.9763 test=4.0296
  seed=2: training...


  seed=2: train=4.0081 val=4.0066 test=4.0074
  seed=3: training...


  seed=3: train=3.9133 val=3.9899 test=4.0223
  seed=4: training...


  seed=4: train=4.0085 val=4.0116 test=4.0108

  seed  train_nll  val_nll  test_nll
    0   3.883516 3.994162  4.006007
    1   3.904200 3.976283  4.029585
    2   4.008068 4.006585  4.007424
    3   3.913346 3.989914  4.022315
    4   4.008527 4.011643  4.010780

mean_test_nll = 4.0152
std_test_nll  = 0.0103


---
## Section 4 — Baseline Verification

Compare against marginal, additive (α-tuned on val), and uniform baselines.
`evaluate_baselines` uses `c["idx_train"]`/`c["idx_test"]` and `c["y_all"]` internally.

In [7]:
# Section 4 — Baseline Verification
baselines = evaluate_baselines(c, X_t_all, N_XT, cfg.alpha_grid, cfg.tau_grid)
uniform_nll = float(np.log(N_actual))  # -log(1/N) = log(N) nats

marginal_nll = -baselines["marginal"]["test_ll"]
additive_nll = -baselines["additive"]["test_ll"]
backoff_nll  = -baselines["backoff"]["test_ll"]
neural_mean  = df_seeds["test_nll"].mean()

print(f"{'Baseline':<20} {'Test NLL':>10}")
print("-" * 32)
print(f"{'Uniform':<20} {uniform_nll:>10.4f}")
print(f"{'Marginal':<20} {marginal_nll:>10.4f}")
print(f"{'Additive':<20} {additive_nll:>10.4f}  (best alpha={baselines['additive']['alpha']})")
print(f"{'Backoff':<20} {backoff_nll:>10.4f}  (best alpha={baselines['backoff']['alpha']}, tau={baselines['backoff']['tau']})")
print(f"{'Neural (mean)':<20} {neural_mean:>10.4f}")

Baseline               Test NLL
--------------------------------
Uniform                  4.0073
Marginal                 4.0073
Additive                 4.0076  (best alpha=10.0)
Backoff                  4.0073  (best alpha=10.0, tau=1000)
Neural (mean)            4.0152


---
## Section 5 — Feature Ablation Confirmation (h=10)

Verify that top-30 MI features is the right operating point for h=10.
Compare feature subset sizes: 15, 30, 50, 194 on test NLL and generalization gap.

In [8]:
# Section 5 — Feature Ablation Training
fa_rows = []
for n_feat in FA_SIZES:
    sel_k = ranking[:n_feat]
    F_k   = F_normed[:, sel_k]
    cache_path = CACHE_DIR / f"fabl_n{n_feat}_h{H}_N{N_BINS}_seed{FA_SEED}.pt"

    random.seed(FA_SEED)
    np.random.seed(FA_SEED)
    torch.manual_seed(FA_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(FA_SEED)

    m_k = StateConditionedNet(
        n_feat=n_feat, n_xt_states=N_XT, n_output=N_actual,
        hidden_dims=cfg.hidden_dims, dropout=cfg.dropout,
    )
    tr_k, val_k, te_k = build_loaders(
        c, F_k, X_t_all,
        batch_train=cfg.batch_train, batch_eval=cfg.batch_eval,
    )

    if is_cached(cache_path):
        print(f"  n_feat={n_feat}: loading from cache...")
        load_cached_model(m_k, cache_path)
    else:
        print(f"  n_feat={n_feat}: training...")
        best_state, _ = train_one_run(
            m_k, tr_k, val_k, N_actual, c["sigma"], DEVICE,
            lr=cfg.lr, weight_decay=cfg.weight_decay,
            max_epochs=cfg.max_epochs, patience=cfg.patience,
            grad_clip=cfg.grad_clip,
        )
        cache_model(best_state, cache_path)

    m_k.to(DEVICE)
    m_k.eval()

    tr_eval_k = DataLoader(
        MasterDataset(F_k, X_t_all, c["y_all"], idx_train),
        batch_size=cfg.batch_eval, shuffle=False,
    )
    r_tr  = evaluate_model(m_k, tr_eval_k, N_actual, DEVICE)
    r_val = evaluate_model(m_k, val_k,     N_actual, DEVICE)
    r_te  = evaluate_model(m_k, te_k,      N_actual, DEVICE)

    fa_rows.append({
        "n_features": n_feat,
        "train_nll":  -r_tr["mean_ll"],
        "val_nll":    -r_val["mean_ll"],
        "test_nll":   -r_te["mean_ll"],
        "gen_gap":    (-r_te["mean_ll"]) - (-r_tr["mean_ll"]),
    })
    print(f"  n_feat={n_feat}: train={fa_rows[-1]['train_nll']:.4f} "
          f"val={fa_rows[-1]['val_nll']:.4f} test={fa_rows[-1]['test_nll']:.4f} "
          f"gap={fa_rows[-1]['gen_gap']:.4f}")

df_fa = pd.DataFrame(fa_rows)
df_fa.to_csv(RESULTS_DIR / "feature_ablation_h10.csv", index=False)
print("\n", df_fa.to_string(index=False))

  n_feat=15: training...


  n_feat=15: train=3.8650 val=3.9697 test=4.1030 gap=0.2380
  n_feat=30: training...


  n_feat=30: train=3.8835 val=3.9942 test=4.0060 gap=0.1225
  n_feat=50: training...


  n_feat=50: train=3.8889 val=3.9896 test=4.0523 gap=0.1634
  n_feat=194: training...


  n_feat=194: train=3.9289 val=3.9952 test=4.0050 gap=0.0761

  n_features  train_nll  val_nll  test_nll  gen_gap
         15   3.865041 3.969693  4.103021 0.237980
         30   3.883516 3.994162  4.006007 0.122491
         50   3.888862 3.989562  4.052288 0.163426
        194   3.928865 3.995208  4.005011 0.076146


In [9]:
# Section 5 — Feature Ablation Plot
fig, ax1 = plt.subplots(figsize=(7, 4))

ax1.plot(df_fa["n_features"], df_fa["test_nll"], "o-b",  label="Test NLL",  lw=2)
ax1.plot(df_fa["n_features"], df_fa["val_nll"],  "s--b", label="Val NLL",   lw=1.5, alpha=0.6)
ax1.set_xlabel("Number of features (top-MI)", fontsize=12)
ax1.set_ylabel("NLL (nats)", color="b", fontsize=12)
ax1.tick_params(axis="y", labelcolor="b")

ax2 = ax1.twinx()
ax2.plot(df_fa["n_features"], df_fa["gen_gap"], "o-r", label="Gen Gap", lw=2, alpha=0.8)
ax2.set_ylabel("Gen Gap (test − train NLL)", color="r", fontsize=12)
ax2.tick_params(axis="y", labelcolor="r")

# Mark optimal point (n=30)
opt_row = df_fa[df_fa["n_features"] == N_FEAT]
if len(opt_row):
    ax1.axvline(N_FEAT, color="gray", ls=":", alpha=0.7, label=f"n={N_FEAT} (selected)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=9)

plt.title(f"Feature Ablation — h={H}, N={N_BINS}", fontsize=13)
plt.tight_layout()
save_fig(fig, FIG_DIR / f"feature_ablation_h{H}_N{N_BINS}")
plt.show()
print(f"Saved: {FIG_DIR / f'feature_ablation_h{H}_N{N_BINS}.png'}")

Saved: /Users/JanRovirosaIlla/DeepMarkovResearch/results/optimal_setup/figures/feature_ablation_h10_N55.png


/var/folders/xp/8yb64l_j1wd31jcqx1l7l_wr0000gp/T/ipykernel_3344/653112190.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Section 6 — Final Summary

Compile key metrics for the paper: mean/std test NLL across seeds,
baseline comparison, and information gain over uniform predictor.

In [10]:
# Section 6 — Final Summary
mean_test_nll    = float(df_seeds["test_nll"].mean())
std_test_nll     = float(df_seeds["test_nll"].std())
additive_nll     = float(-baselines["additive"]["test_ll"])
marginal_nll     = float(-baselines["marginal"]["test_ll"])
backoff_nll      = float(-baselines["backoff"]["test_ll"])
uniform_nll      = float(np.log(N_actual))
information_gain = uniform_nll - mean_test_nll  # nats below uniform (positive = better)

summary = pd.DataFrame([
    {"metric": "mean_test_nll",     "value": mean_test_nll},
    {"metric": "std_test_nll",      "value": std_test_nll},
    {"metric": "uniform_nll",       "value": uniform_nll},
    {"metric": "marginal_nll",      "value": marginal_nll},
    {"metric": "additive_nll",      "value": additive_nll},
    {"metric": "backoff_nll",       "value": backoff_nll},
    {"metric": "information_gain",  "value": information_gain},
])
summary.to_csv(RESULTS_DIR / "final_summary.csv", index=False)

print("=" * 40)
print("OPTIMAL SETUP FINAL RESULTS")
print(f"  h={H}, N={N_BINS}, n_feat={N_FEAT}")
print(f"  seeds={SEEDS}")
print("=" * 40)
print(summary.to_string(index=False))
print("=" * 40)
print(f"\nNeural vs Additive:  ΔNLL = {mean_test_nll - additive_nll:+.4f}")
print(f"Neural vs Marginal:  ΔNLL = {mean_test_nll - marginal_nll:+.4f}")
print(f"Information gain over uniform: {information_gain:.4f} nats")

OPTIMAL SETUP FINAL RESULTS
  h=10, N=55, n_feat=30
  seeds=[0, 1, 2, 3, 4]
          metric     value
   mean_test_nll  4.015222
    std_test_nll  0.010272
     uniform_nll  4.007333
    marginal_nll  4.007333
    additive_nll  4.007645
     backoff_nll  4.007273
information_gain -0.007889

Neural vs Additive:  ΔNLL = +0.0076
Neural vs Marginal:  ΔNLL = +0.0079
Information gain over uniform: -0.0079 nats
